# SAM3 Auto Point Selection for MapAnything Measurement

这个notebook只处理 **SAM3 segmentation + automatic point proposal**。  
它不会重新跑MapAnything。默认会复用之前notebook已经生成的image folder和MapAnything `.npy` output。

目标：

1. 用SAM3的text prompt自动找 `road sign / stop sign / traffic sign` mask。
2. 从mask里自动生成measurement candidate points，比如sign bottom、sign top、sign center。
3. 如果已有MapAnything `pts3d / mask / image` output，则把2D point自动scale到MapAnything output resolution，snap到nearest valid 3D point，然后计算3D distance。
4. 导出overlay、mask、candidate point CSV，方便和之前manual point selection对比。

In [1]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from torchvision.ops import nms
print("torchvision.ops.nms OK")

ModuleNotFoundError: No module named 'torch'

In [10]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U triton timm huggingface_hub

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 24.9 MB/s eta 0:00:00
ERROR: Operation cancelled by user
^C


In [ ]:
import torch
import torchvision
import triton

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Triton:", triton.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from torchvision.ops import nms
print("torchvision.ops.nms OK")

In [ ]:
# 0. Basic setup

import os, sys, gc, json, math, shutil, subprocess, importlib.util
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is required before running SAM3.") from exc

PROJECT_ROOT = Path.cwd()

# Change these only if your previous notebook used different folders.
EXPERIMENT_ROOT = Path("/content/mapillary_idot_pilot")
SINGLE_IMAGE_DIR = EXPERIMENT_ROOT / "vertical_clearance_selected" / "single_input"
MULTI_IMAGE_DIR = EXPERIMENT_ROOT / "vertical_clearance_selected" / "multi_input"

# Existing MapAnything output from the previous notebook.
MAPANYTHING_SINGLE_OUT = EXPERIMENT_ROOT / "single_0m_3d_output"

# New SAM3-only output folder.
SAM3_OUT = EXPERIMENT_ROOT / "sam3_auto_point_selection"
MASK_DIR = SAM3_OUT / "masks"
VIS_DIR = SAM3_OUT / "visualizations"
for d in [SAM3_OUT, MASK_DIR, VIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_ID = "842149990032290"
IMAGE_PATH = SINGLE_IMAGE_DIR / f"{IMAGE_ID}.jpg"

SAM3_TEXT_PROMPTS = [
    "stop sign",
    "road sign",
    "traffic sign",
    "speed limit sign",
    "warning road sign",
    "street sign",
]

MAX_OBJECTS_PER_IMAGE = 8
MIN_SCORE = 0.25
FINAL_SCORE = 0.50
print("CUDA:", torch.cuda.is_available())
print("Image path:", IMAGE_PATH, "exists=", IMAGE_PATH.exists())
print("MapAnything output folder:", MAPANYTHING_SINGLE_OUT, "exists=", MAPANYTHING_SINGLE_OUT.exists())

## 1. Install / load SAM3

Run this cell only when `import sam3` fails.  
SAM3 weights are gated on Hugging Face, so you need approved access to `facebook/sam3` and a logged-in HF token. In Colab, run:

```bash
huggingface-cli login
```

Then restart runtime if imports still fail.

In [ ]:
# 1. Optional install

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

if not has_module("sam3"):
    print("sam3 is not installed. Installing from facebookresearch/sam3...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/sam3.git"])
else:
    print("sam3 already installed.")

try:
    from huggingface_hub import whoami
    print("HF login:", whoami().get("name", "logged-in"))
except Exception as exc:
    print("HF login not detected or token not valid.")
    print("Run: huggingface-cli login")
    print(type(exc).__name__, str(exc)[:200])

In [ ]:
import os
os.environ["MAPILLARY_TOKEN"] = "<YOUR_MAPILLARY_TOKEN>"

In [ ]:
# 2. Load / download image

import os
import requests
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

IMAGE_ID = "842149990032290"

WORK_DIR = Path("/content/sam3_auto_point_selection")
IMAGE_DIR = WORK_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_PATH = IMAGE_DIR / f"{IMAGE_ID}.jpg"

# Option A: use local image if it already exists
if not IMAGE_PATH.exists():
    # Option B: download from Mapillary
    MAPILLARY_TOKEN = os.environ.get("MAPILLARY_TOKEN", "")

    if not MAPILLARY_TOKEN:
        raise ValueError(
            "MAPILLARY_TOKEN is missing. In Colab, run:\n"
            "import os\n"
            "os.environ['MAPILLARY_TOKEN'] = 'your_mapillary_token_here'"
        )

    meta_url = f"https://graph.mapillary.com/{IMAGE_ID}"
    params = {
        "fields": "thumb_2048_url,thumb_original_url,width,height",
        "access_token": MAPILLARY_TOKEN,
    }

    r = requests.get(meta_url, params=params)
    r.raise_for_status()
    meta = r.json()

    img_url = meta.get("thumb_original_url") or meta.get("thumb_2048_url")
    if img_url is None:
        raise RuntimeError(f"No downloadable image URL found for image_id={IMAGE_ID}")

    img_bytes = requests.get(img_url)
    img_bytes.raise_for_status()

    IMAGE_PATH.write_bytes(img_bytes.content)

image = Image.open(IMAGE_PATH).convert("RGB")

print("Loaded:", IMAGE_PATH)
print("Image size:", image.size)

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
# 3. SAM3 helpers copied/adapted from the executed notebook

@dataclass
class Detection:
    prompt: str
    score: float
    bbox_xyxy: Tuple[int, int, int, int]
    mask: np.ndarray

def tensor_to_numpy(x):
    if hasattr(x, "detach"):
        if getattr(x, "is_floating_point", lambda: False)():
            x = x.float()
        return x.detach().cpu().numpy()
    return np.asarray(x)

def box_iou_xyxy(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0

def nms_detections(dets: List[Detection], iou_threshold: float = 0.70) -> List[Detection]:
    kept = []
    for det in sorted(dets, key=lambda d: d.score, reverse=True):
        if all(box_iou_xyxy(det.bbox_xyxy, old.bbox_xyxy) < iou_threshold for old in kept):
            kept.append(det)
    return kept

def road_sign_filter(dets: List[Detection], image: Image.Image) -> List[Detection]:
    w_img, h_img = image.size
    selected = []
    for d in dets:
        x1, y1, x2, y2 = d.bbox_xyxy
        bw, bh = x2 - x1, y2 - y1
        area = bw * bh
        cy = (y1 + y2) / 2
        aspect = bw / max(bh, 1)
        if d.score < FINAL_SCORE:
            continue
        if area < 0.00015 * w_img * h_img:
            continue
        if area > 0.20 * w_img * h_img:
            continue
        if not (0.03 * h_img <= cy <= 0.95 * h_img):
            continue
        if not (0.15 <= aspect <= 6.00):
            continue
        selected.append(d)
    return sorted(selected, key=lambda d: (-d.score, d.bbox_xyxy[1], d.bbox_xyxy[0]))[:MAX_OBJECTS_PER_IMAGE]

In [ ]:
!pip install -q triton

In [ ]:
!pip install -q -U torch torchvision torchaudio

In [ ]:
# 4. Run SAM3 text-prompt segmentation

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = build_sam3_image_model().to(device)
model.eval()
processor = Sam3Processor(model)

raw_dets = []
with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
        state = processor.set_image(image)
        for prompt in SAM3_TEXT_PROMPTS:
            out = processor.set_text_prompt(state=state, prompt=prompt)
            masks = tensor_to_numpy(out["masks"])
            boxes = tensor_to_numpy(out["boxes"])
            scores = tensor_to_numpy(out["scores"])
            print(prompt, "objects:", len(scores))
            for k, score in enumerate(scores):
                score = float(score)
                if score < MIN_SCORE:
                    continue
                mask = masks[k]
                if mask.ndim == 3:
                    mask = mask.squeeze()
                raw_dets.append(
                    Detection(
                        prompt=prompt,
                        score=score,
                        bbox_xyxy=tuple(map(int, boxes[k].tolist())),
                        mask=mask.astype(bool),
                    )
                )

sam3_detections = road_sign_filter(nms_detections(raw_dets), image)
print("Raw detections:", len(raw_dets))
print("Final detections:", len(sam3_detections))

del processor, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 5. Visualize and save SAM3 masks

def overlay_detections(image: Image.Image, detections: List[Detection]) -> Image.Image:
    base = image.convert("RGBA")
    for i, d in enumerate(detections):
        mask = d.mask
        if mask.shape != (image.height, image.width):
            mask = np.asarray(
                Image.fromarray(mask.astype(np.uint8) * 255).resize(image.size, Image.Resampling.NEAREST)
            ) > 0
        overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
        color = Image.new("RGBA", image.size, (255, 40, 40, 120))
        overlay.paste(color, mask=Image.fromarray(mask.astype(np.uint8) * 255))
        base = Image.alpha_composite(base, overlay)
        draw = ImageDraw.Draw(base)
        draw.rectangle(d.bbox_xyxy, outline=(255, 0, 0, 255), width=5)
        x1, y1, _, _ = d.bbox_xyxy
        draw.text((x1, max(0, y1 - 22)), f"{i}: {d.prompt} {d.score:.2f}", fill=(255, 0, 0, 255))
    return base.convert("RGB")

overlay = overlay_detections(image, sam3_detections)
overlay_path = VIS_DIR / f"{IMAGE_PATH.stem}_sam3_overlay.jpg"
overlay.save(overlay_path)

rows = []
for i, d in enumerate(sam3_detections):
    mask_path = MASK_DIR / f"{IMAGE_PATH.stem}_det{i:02d}_{d.prompt.replace(' ', '_')}.png"
    Image.fromarray(d.mask.astype(np.uint8) * 255).save(mask_path)
    rows.append({
        "det_idx": i,
        "prompt": d.prompt,
        "score": d.score,
        "bbox_xyxy": d.bbox_xyxy,
        "mask_path": str(mask_path),
        "area_px": int(d.mask.sum()),
    })

detections_df = pd.DataFrame(rows)
detections_df.to_csv(SAM3_OUT / "sam3_detections.csv", index=False)
display(detections_df)

plt.figure(figsize=(9, 7))
plt.imshow(overlay)
plt.axis("off")
plt.show()
print("Saved:", overlay_path)

In [ ]:
# 6. Convert SAM3 masks into automatic 2D measurement point proposals

def points_from_mask(mask: np.ndarray, bbox: Tuple[int, int, int, int]) -> Dict[str, Tuple[int, int]]:
    ys, xs = np.where(mask)
    if len(xs) < 10:
        x1, y1, x2, y2 = bbox
        cx = int(round((x1 + x2) / 2))
        return {
            "top": (cx, int(y1)),
            "bottom": (cx, int(y2)),
            "center": (cx, int(round((y1 + y2) / 2))),
        }

    # Robust edge points: median x among top/bottom mask bands.
    top_y = int(np.percentile(ys, 5))
    bottom_y = int(np.percentile(ys, 95))
    center_y = int(np.percentile(ys, 50))

    top_band = np.abs(ys - top_y) <= 3
    bottom_band = np.abs(ys - bottom_y) <= 3
    center_band = np.abs(ys - center_y) <= 3

    def band_x(sel):
        return int(np.median(xs[sel])) if np.any(sel) else int(np.median(xs))

    return {
        "top": (band_x(top_band), top_y),
        "bottom": (band_x(bottom_band), bottom_y),
        "center": (band_x(center_band), center_y),
    }

point_rows = []
point_vis = image.copy()
draw = ImageDraw.Draw(point_vis)

for i, d in enumerate(sam3_detections):
    pts = points_from_mask(d.mask, d.bbox_xyxy)
    for name, (x, y) in pts.items():
        point_rows.append({"det_idx": i, "prompt": d.prompt, "point_name": name, "x": x, "y": y})
        r = 8
        draw.ellipse((x-r, y-r, x+r, y+r), outline=(255, 0, 0), width=4)
        draw.text((x + 10, y - 10), f"{i}:{name}", fill=(255, 0, 0))
    draw.line([pts["top"], pts["bottom"]], fill=(255, 0, 0), width=4)

points_df = pd.DataFrame(point_rows)
points_df.to_csv(SAM3_OUT / "sam3_auto_2d_points.csv", index=False)

point_vis_path = VIS_DIR / f"{IMAGE_PATH.stem}_sam3_auto_points.jpg"
point_vis.save(point_vis_path)
display(points_df)

plt.figure(figsize=(9, 7))
plt.imshow(point_vis)
plt.axis("off")
plt.show()
print("Saved:", point_vis_path)

In [ ]:
# 7. Optional: load previous MapAnything output without rerunning MapAnything

def load_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return np.load(p, allow_pickle=True)
    return None

single_pts3d = load_first_existing([
    MAPANYTHING_SINGLE_OUT / "single_pts3d.npy",
    MAPANYTHING_SINGLE_OUT / "pts3d.npy",
])

single_valid = load_first_existing([
    MAPANYTHING_SINGLE_OUT / "single_mask.npy",
    MAPANYTHING_SINGLE_OUT / "mask.npy",
])

single_img_model = load_first_existing([
    MAPANYTHING_SINGLE_OUT / "single_img_model.npy",
    MAPANYTHING_SINGLE_OUT / "img_model.npy",
])

print("pts3d:", None if single_pts3d is None else single_pts3d.shape)
print("valid mask:", None if single_valid is None else single_valid.shape)
print("model image:", None if single_img_model is None else single_img_model.shape)

if single_pts3d is None or single_valid is None:
    print("MapAnything output not found. SAM3 2D points are still saved; 3D measurement is skipped.")

In [ ]:
# 8. Snap SAM3 2D points to nearest valid MapAnything 3D points and measure top-bottom distance

def snap_to_nearest_valid(x, y, valid_mask, max_radius=25):
    h, w = valid_mask.shape
    x = int(np.clip(x, 0, w - 1))
    y = int(np.clip(y, 0, h - 1))
    if valid_mask[y, x]:
        return x, y, 0
    yy, xx = np.where(valid_mask)
    if len(xx) == 0:
        return x, y, None
    dist2 = (xx - x) ** 2 + (yy - y) ** 2
    j = int(np.argmin(dist2))
    dist = float(np.sqrt(dist2[j]))
    if dist > max_radius:
        return int(xx[j]), int(yy[j]), dist
    return int(xx[j]), int(yy[j]), dist

def scale_xy_to_model(x, y, orig_size, model_hw):
    orig_w, orig_h = orig_size
    mh, mw = model_hw
    return int(round(x * mw / orig_w)), int(round(y * mh / orig_h))

measurement_rows = []

if single_pts3d is not None and single_valid is not None and len(points_df):
    pts3d = np.asarray(single_pts3d)
    valid = np.asarray(single_valid).astype(bool)
    if pts3d.ndim == 4:
        pts3d = pts3d[0]
    if valid.ndim == 3:
        valid = valid[0]

    Hm, Wm = pts3d.shape[:2]
    for det_idx, g in points_df.groupby("det_idx"):
        point_map = {r.point_name: (int(r.x), int(r.y)) for r in g.itertuples()}
        if "top" not in point_map or "bottom" not in point_map:
            continue

        coords3d = {}
        snap_info = {}
        for name in ["top", "bottom", "center"]:
            if name not in point_map:
                continue
            xm, ym = scale_xy_to_model(*point_map[name], image.size, (Hm, Wm))
            xs, ys, shift = snap_to_nearest_valid(xm, ym, valid)
            coords3d[name] = pts3d[ys, xs]
            snap_info[name] = {"model_x": xm, "model_y": ym, "snap_x": xs, "snap_y": ys, "snap_shift_px": shift}

        dist_m = float(np.linalg.norm(coords3d["top"] - coords3d["bottom"]))
        prompt = str(g["prompt"].iloc[0])
        measurement_rows.append({
            "det_idx": int(det_idx),
            "prompt": prompt,
            "auto_distance_m_top_to_bottom": dist_m,
            "auto_distance_ft_top_to_bottom": dist_m * 3.280839895,
            "top_original_xy": point_map["top"],
            "bottom_original_xy": point_map["bottom"],
            "top_snap_info": snap_info["top"],
            "bottom_snap_info": snap_info["bottom"],
        })

auto_measurement_df = pd.DataFrame(measurement_rows)
auto_measurement_df.to_csv(SAM3_OUT / "sam3_auto_mapanything_measurements.csv", index=False)
display(auto_measurement_df)

## 9. How to use this for replacing manual point selection

For your current vertical-clearance experiment, SAM3 should not directly replace every manual point with one fixed rule. Better workflow:

1. Use SAM3 to automatically detect the sign/object mask.
2. Use mask-derived `top`, `bottom`, and `center` as candidate object points.
3. Use MapAnything valid 3D mask to snap those points to valid 3D locations.
4. For clearance, use `bottom` of the sign/object as the upper object point.
5. For the road/ground point, either:
   - keep one manual road point for now, or
   - fit a local road plane from MapAnything/Depth output and compute sign-bottom-to-plane distance.

This notebook implements steps 1–3. Step 5 can be added after confirming SAM3 finds the correct object on your Mapillary image.

In [ ]:
# 10. Summary paths

print("SAM3 output folder:", SAM3_OUT)
print("Detections CSV:", SAM3_OUT / "sam3_detections.csv")
print("2D points CSV:", SAM3_OUT / "sam3_auto_2d_points.csv")
print("3D measurement CSV:", SAM3_OUT / "sam3_auto_mapanything_measurements.csv")
print("Overlay:", VIS_DIR / f"{IMAGE_PATH.stem}_sam3_overlay.jpg")
print("Auto points:", VIS_DIR / f"{IMAGE_PATH.stem}_sam3_auto_points.jpg")